# 0d Heatbath — Casseau (2015)

Set up a heatbath with only one periodic cell and compare against the Casseau reference datasets. The notebook interface follows the same pattern as the Williams case: shared setup helpers first, then one direct user-input run cell and one plot cell per case.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

from pathlib import Path
from typing import Sequence

import jax
import jax.numpy as jnp
import pandas as pd

import compressible.chemistry_types as chemistry_types
import compressible.chemistry_utils as chemistry_utils
import compressible.constants as constants
import compressible.energy_models as energy_models
from compressible.boundary_conditions_utils import build_boundary_arrays_1d_periodic
from compressible.equation_manager import run_scan
from compressible.equation_manager_types import EquationManager
from compressible.mesh import Mesh
from compressible.numerics_types import ClippingConfig, NumericsConfig
from compressible.state import compute_U_from_primitives, extract_primitives_from_U

import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"

In [2]:
REPO_ROOT = Path("/home/hhoechter/tum/jaxfluids_internship")
DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "experiments" / "heatbath_0d_casseau"

ENERGY_DATA_PATHS = {
    "bird": DATA_DIR / "air_5_bird_energy.json",
    "gnoffo": DATA_DIR / "air_5_gnoffo_equilibrium_enthalpy.json",
}
REACTION_DATA_PATHS = {
    "park": DATA_DIR / "park_reactions.json",
    "qk": DATA_DIR / "casseau_qk_reactions.json",
    "n2_park": RESULTS_DIR / "n2_reaction_set_park.json",
    "n2_qk": RESULTS_DIR / "n2_reaction_set_qk.json",
}
CHEMISTRY_MODEL_CONFIGS = {
    "park_pref": dict(model="park", park_vibrational_source="preferential_constant"),
    "park_nonpref": dict(model="park", park_vibrational_source="nonpreferential"),
    "cvdv_qp": dict(model="cvdv_qp"),
}
MODEL_DISPLAY_NAMES = {
    "park_pref": "Park preferential",
    "park_nonpref": "Park nonpreferential",
    "cvdv_qp": "CVDV-QP",
}
MODEL_COLORS = {
    "park_pref": "#1f77b4",
    "park_nonpref": "#d62728",
    "cvdv_qp": "#2ca02c",
}
REF_COLORS = ["#9467bd", "#ff7f0e", "#8c564b", "#e377c2", "#7f7f7f"]


def _extract_prim(U, equation_manager):
    prim = extract_primitives_from_U(U, equation_manager)
    return prim.Y_s, prim.rho, prim.T, prim.Tv, prim.p


extract_primitives_from_U_jitted = jax.jit(_extract_prim)


def normalize_rows(values: jnp.ndarray) -> jnp.ndarray:
    values = jnp.asarray(values)
    if values.ndim == 1:
        values = values[None, :]
    return values / jnp.clip(jnp.sum(values, axis=1, keepdims=True), 1e-14, None)


def load_species_table(
    species_names: Sequence[str],
    energy_model: str,
    include_electronic: bool,
) -> chemistry_utils.SpeciesTable:
    energy_cfg = energy_models.EnergyModelConfig(
        model=energy_model,
        include_electronic=include_electronic,
        data_path=str(ENERGY_DATA_PATHS[energy_model]),
    )
    return chemistry_utils.load_species_table(
        species_names=species_names,
        general_data_path=str(DATA_DIR / "air_5_gnoffo.json"),
        energy_model_config=energy_cfg,
    )


def build_time_controls(
    *,
    time_mode: str,
    dt: float | None = None,
    t_final: float,
    dt_fine: float | None = None,
    dt_coarse: float | None = None,
    t_threshold: float | None = None,
) -> tuple[float, float, jnp.ndarray | None]:
    if time_mode == "fixed":
        if dt is None:
            raise ValueError("time_mode='fixed' requires dt")
        return float(dt), float(t_final), None
    if time_mode == "two_phase":
        if dt_fine is None or dt_coarse is None or t_threshold is None:
            raise ValueError(
                "time_mode='two_phase' requires dt_fine, dt_coarse, and t_threshold"
            )
        n_fine = int(t_threshold / dt_fine)
        n_coarse = int((t_final - t_threshold) / dt_coarse)
        dt_array = jnp.concatenate(
            [
                jnp.full((n_fine,), dt_fine),
                jnp.full((n_coarse,), dt_coarse),
            ]
        )
        return float(dt_fine), float(t_final), dt_array
    raise ValueError(f"Unknown time_mode: {time_mode!r}")


def build_equation_manager(
    *,
    species,
    reactions,
    dx: float,
    dt: float,
    integrator_scheme: str,
) -> tuple[Mesh, EquationManager]:
    mesh = Mesh.from_1d_grid(jnp.array([0.0, dx]), periodic=True)
    boundary_arrays = build_boundary_arrays_1d_periodic(mesh, species.n_species)
    numerics_config = NumericsConfig(
        dt=dt,
        cfl=0.4,
        dt_mode="fixed",
        integrator_scheme=integrator_scheme,
        spatial_scheme="first_order",
        flux_scheme="hllc",
        clipping=ClippingConfig(),
    )
    equation_manager = EquationManager(
        species=species,
        reactions=reactions,
        numerics_config=numerics_config,
        boundary_arrays=boundary_arrays,
    )
    return mesh, equation_manager


def build_U_from_mass_fractions(
    *,
    equation_manager: EquationManager,
    species_names: Sequence[str],
    composition: dict[str, float],
    T_tr_init: float,
    T_V_init: float,
    p_init_atm: float,
) -> jnp.ndarray:
    species = equation_manager.species
    p_init_pa = p_init_atm * constants.ATM_TO_PA
    species_names = list(species_names)
    Y_init = normalize_rows(jnp.array([composition[name] for name in species_names]))
    molar_masses = jnp.array(
        [species.molar_masses[species.names.index(name)] for name in species_names]
    )
    M_mix = jnp.sum(Y_init * molar_masses[None, :], axis=1)[0]
    rho_init = p_init_pa * float(M_mix) / (constants.R_universal * T_tr_init)
    return compute_U_from_primitives(
        Y_s=Y_init,
        rho=jnp.array([rho_init]),
        u=jnp.array([0.0]),
        v=jnp.zeros(1),
        T_tr=jnp.array([T_tr_init]),
        T_V=jnp.array([T_V_init]),
        equation_manager=equation_manager,
    )


def build_U_from_number_densities(
    *,
    equation_manager: EquationManager,
    species_names: Sequence[str],
    number_densities: dict[str, float],
    T_tr_init: float,
    T_V_init: float,
    p_init_atm: float | None = None,
) -> tuple[jnp.ndarray, float]:
    species = equation_manager.species
    species_names = list(species_names)
    n_values = jnp.array([number_densities.get(name, 0.0) for name in species_names])
    Y_init = normalize_rows(n_values)
    molar_masses = jnp.array(
        [species.molar_masses[species.names.index(name)] for name in species_names]
    )
    # When p_init_atm is set, use number_densities only to define composition and
    # initialize the legacy heatbath benchmark at the requested pressure.
    if p_init_atm is None:
        n_total = float(jnp.sum(n_values))
        rho_init = 0.0
        for name in species_names:
            idx = species.names.index(name)
            rho_init += (
                number_densities.get(name, 0.0)
                * float(species.molar_masses[idx])
                / constants.N_A
            )
    else:
        M_mix = jnp.sum(Y_init * molar_masses[None, :], axis=1)[0]
        rho_init = (
            p_init_atm
            * constants.ATM_TO_PA
            * float(M_mix)
            / (constants.R_universal * T_tr_init)
        )
        n_total = (
            p_init_atm
            * constants.ATM_TO_PA
            / (constants.R_universal * T_tr_init)
            * constants.N_A
        )
    U_init = compute_U_from_primitives(
        Y_s=Y_init,
        rho=jnp.array([rho_init]),
        u=jnp.array([0.0]),
        v=jnp.zeros(1),
        T_tr=jnp.array([T_tr_init]),
        T_V=jnp.array([T_V_init]),
        equation_manager=equation_manager,
    )
    return U_init, n_total


def run_single_heatbath_case(
    *,
    species_names: Sequence[str],
    energy_model: str,
    include_electronic: bool,
    T_tr_init: float,
    T_V_init: float,
    dx: float,
    save_interval: int,
    time_mode: str,
    t_final: float,
    integrator_scheme: str = "forward-euler",
    dt: float | None = None,
    dt_fine: float | None = None,
    dt_coarse: float | None = None,
    t_threshold: float | None = None,
    p_init_atm: float | None = None,
    mass_fraction_composition: dict[str, float] | None = None,
    number_densities: dict[str, float] | None = None,
) -> dict:
    dt0, t_final_value, dt_array = build_time_controls(
        time_mode=time_mode,
        dt=dt,
        t_final=t_final,
        dt_fine=dt_fine,
        dt_coarse=dt_coarse,
        t_threshold=t_threshold,
    )
    species = load_species_table(species_names, energy_model, include_electronic)
    mesh, equation_manager = build_equation_manager(
        species=species,
        reactions=None,
        dx=dx,
        dt=dt0,
        integrator_scheme=integrator_scheme,
    )
    n_total_init = None
    if mass_fraction_composition is not None:
        if p_init_atm is None:
            raise ValueError("p_init_atm is required for mass_fraction_composition")
        U_init = build_U_from_mass_fractions(
            equation_manager=equation_manager,
            species_names=species_names,
            composition=mass_fraction_composition,
            T_tr_init=T_tr_init,
            T_V_init=T_V_init,
            p_init_atm=p_init_atm,
        )
        n_total_init = (
            p_init_atm
            * constants.ATM_TO_PA
            / (constants.R_universal * T_tr_init)
            * constants.N_A
        )
    elif number_densities is not None:
        U_init, n_total_init = build_U_from_number_densities(
            equation_manager=equation_manager,
            species_names=species_names,
            number_densities=number_densities,
            T_tr_init=T_tr_init,
            T_V_init=T_V_init,
            p_init_atm=p_init_atm,
        )
    else:
        raise ValueError("Provide either mass_fraction_composition or number_densities")

    U_hist, t_hist = run_scan(
        U_init=U_init,
        mesh=mesh,
        equation_manager=equation_manager,
        t_final=t_final_value,
        save_interval=save_interval,
        dt_array=dt_array,
    )
    Y_s, rho, T, T_V, p = jax.vmap(extract_primitives_from_U_jitted, in_axes=(0, None))(
        U_hist, equation_manager
    )
    mixture_molar_mass = jnp.sum(Y_s[:, 0, :] * species.molar_masses[None, :], axis=1)
    n_species_hist = Y_s[:, 0, :] * rho / mixture_molar_mass[:, None] * constants.N_A
    return {
        "species": species,
        "equation_manager": equation_manager,
        "U_hist": U_hist,
        "t": t_hist,
        "Y_s": Y_s,
        "rho": rho,
        "T": T,
        "T_V": T_V,
        "p": p,
        "n_species": n_species_hist,
        "n_total_init": n_total_init,
    }


def run_reacting_comparison(
    *,
    species_names: Sequence[str],
    number_densities: dict[str, float],
    T_tr_init: float,
    T_V_init: float,
    energy_model: str,
    include_electronic: bool,
    comparison_models: Sequence[dict],
    dx: float,
    save_interval: int,
    time_mode: str,
    t_final: float,
    dt: float | None = None,
    dt_fine: float | None = None,
    dt_coarse: float | None = None,
    t_threshold: float | None = None,
) -> dict[str, dict]:
    dt0, t_final_value, dt_array = build_time_controls(
        time_mode=time_mode,
        dt=dt,
        t_final=t_final,
        dt_fine=dt_fine,
        dt_coarse=dt_coarse,
        t_threshold=t_threshold,
    )
    results = {}
    for model_spec in comparison_models:
        chemistry_model = model_spec["chemistry_model"]
        reaction_set = model_spec["reaction_set"]
        model_energy = model_spec.get("energy_model", energy_model)
        model_include_electronic = model_spec.get(
            "include_electronic", include_electronic
        )
        species = load_species_table(
            species_names, model_energy, model_include_electronic
        )
        chemistry_model_config = chemistry_types.ChemistryModelConfig(
            **CHEMISTRY_MODEL_CONFIGS[chemistry_model]
        )
        reactions = chemistry_utils.load_reactions_from_json(
            json_path=str(REACTION_DATA_PATHS[reaction_set]),
            species_table=species,
            chemistry_model_config=chemistry_model_config,
        )
        included_reactions, excluded_reactions = (
            chemistry_utils.check_reaction_coverage(
                json_path=str(REACTION_DATA_PATHS[reaction_set]),
                species_names=species.names,
            )
        )
        mesh, equation_manager = build_equation_manager(
            species=species,
            reactions=reactions,
            dx=dx,
            dt=dt0,
            integrator_scheme="forward-euler",
        )
        U_init, n_total_init = build_U_from_number_densities(
            equation_manager=equation_manager,
            species_names=species_names,
            number_densities=number_densities,
            T_tr_init=T_tr_init,
            T_V_init=T_V_init,
        )
        U_hist, t_hist = run_scan(
            U_init=U_init,
            mesh=mesh,
            equation_manager=equation_manager,
            t_final=t_final_value,
            save_interval=save_interval,
            dt_array=dt_array,
        )
        Y_s, rho, T, T_V, p = jax.vmap(
            extract_primitives_from_U_jitted, in_axes=(0, None)
        )(U_hist, equation_manager)
        mixture_molar_mass = jnp.sum(
            Y_s[:, 0, :] * species.molar_masses[None, :], axis=1
        )
        n_species_hist = (
            Y_s[:, 0, :] * rho / mixture_molar_mass[:, None] * constants.N_A
        )
        results[chemistry_model] = {
            "display_name": MODEL_DISPLAY_NAMES.get(chemistry_model, chemistry_model),
            "color": MODEL_COLORS.get(chemistry_model, "#000000"),
            "reaction_set": reaction_set,
            "energy_model": model_energy,
            "include_electronic": model_include_electronic,
            "included_reactions": included_reactions,
            "excluded_reactions": excluded_reactions,
            "species": species,
            "equation_manager": equation_manager,
            "U_hist": U_hist,
            "t": t_hist,
            "Y_s": Y_s,
            "rho": rho,
            "T": T,
            "T_V": T_V,
            "p": p,
            "n_species": n_species_hist,
            "n_total_init": n_total_init,
        }
        print(
            f"Completed {chemistry_model}: reaction_set={reaction_set}, energy_model={model_energy}"
        )
    return results


def print_comparison_summary(results: dict[str, dict]) -> None:
    for chemistry_model, result in results.items():
        print()
        print(f"=== {result['display_name']} ({chemistry_model}) ===")
        print("Included reactions:")
        for rxn in result["included_reactions"]:
            print(f"  {rxn['equation']}")
        print("Excluded reactions:")
        for rxn in result["excluded_reactions"]:
            print(f"  {rxn['equation']} - Missing: {list(rxn['missing_species'])}")


def load_reference_csv(csv_path: str | Path):
    df = pd.read_csv(csv_path, skiprows=1)
    with open(csv_path, "r", encoding="utf-8") as handle:
        dataset_names = [name for name in handle.readline().strip().split(",") if name]
    return df, dataset_names


def dataset_prefix(name: str) -> str:
    return name.replace("_t_v", "").replace("_t_tr", "").replace("_density", "")


def iter_reference_traces(csv_path: str | Path):
    df, dataset_names = load_reference_csv(csv_path)
    for i, name in enumerate(dataset_names):
        x_col = i * 2
        y_col = i * 2 + 1
        if y_col >= len(df.columns):
            continue
        x_data = pd.to_numeric(df.iloc[:, x_col], errors="coerce").dropna().values
        y_data = pd.to_numeric(df.iloc[:, y_col], errors="coerce").dropna().values
        n = min(len(x_data), len(y_data))
        yield name, x_data[:n], y_data[:n]


def make_temperature_reference_plot(
    *,
    result: dict,
    reference_csv: str | Path,
    title: str,
    yaxis_range: list[float] | None = None,
    xaxis_range: list[float] | None = None,
    T_eq: float | None = None,
):
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(x=result["t"], y=result["T_V"][:, 0], mode="lines", name="T_V")
    )
    fig.add_trace(
        go.Scatter(x=result["t"], y=result["T"][:, 0], mode="lines", name="T")
    )
    if T_eq is not None:
        fig.add_hline(y=T_eq, line_dash="dash", line_color="black")
    prefix_colors = {}
    for i, (name, x_data, y_data) in enumerate(iter_reference_traces(reference_csv)):
        prefix = dataset_prefix(name)
        color = prefix_colors.setdefault(
            prefix, REF_COLORS[len(prefix_colors) % len(REF_COLORS)]
        )
        fig.add_trace(
            go.Scatter(
                x=x_data,
                y=y_data * 1000.0,
                mode="markers+lines",
                name=name,
                line=dict(dash="dot", shape="spline", smoothing=1.0),
                marker=dict(
                    color=color,
                    symbol="circle"
                    if "_t_v" in name
                    else "square"
                    if "_t_tr" in name
                    else "circle",
                    size=10,
                ),
            )
        )
    fig.update_layout(
        template="simple_white",
        title=title,
        xaxis_title="Time (s)",
        yaxis_title="Temperature (K)",
    )
    fig.update_xaxes(
        type="log", exponentformat="power", showexponent="all", showgrid=True
    )
    fig.update_yaxes(showgrid=True)
    if yaxis_range is not None:
        fig.update_yaxes(range=yaxis_range)
    if xaxis_range is not None:
        fig.update_xaxes(range=xaxis_range)
    fig.show()


def make_reacting_temperature_comparison_plot(
    *,
    results: dict[str, dict],
    reference_csv: str | Path,
    title: str,
    yaxis_range: list[float] | None = None,
    xaxis_range: list[float] | None = None,
    subsample_start_index: int = 10,
    subsample_factor: int = 10,
):
    fig = go.Figure()
    for chemistry_model, result in results.items():
        t_plot = jnp.concatenate(
            [
                result["t"][:subsample_start_index],
                result["t"][subsample_start_index::subsample_factor],
            ],
            axis=0,
        )
        T_plot = jnp.concatenate(
            [
                result["T"][:subsample_start_index],
                result["T"][subsample_start_index::subsample_factor],
            ],
            axis=0,
        )
        Tv_plot = jnp.concatenate(
            [
                result["T_V"][:subsample_start_index],
                result["T_V"][subsample_start_index::subsample_factor],
            ],
            axis=0,
        )
        color = result["color"]
        fig.add_trace(
            go.Scatter(
                x=t_plot,
                y=Tv_plot[:, 0],
                mode="lines",
                name=f"{result['display_name']} T_V",
                line=dict(color=color),
            )
        )
        fig.add_trace(
            go.Scatter(
                x=t_plot,
                y=T_plot[:, 0],
                mode="lines",
                name=f"{result['display_name']} T",
                line=dict(color=color, dash="dash"),
            )
        )
    prefix_colors = {}
    for name, x_data, y_data in iter_reference_traces(reference_csv):
        prefix = dataset_prefix(name)
        color = prefix_colors.setdefault(
            prefix, REF_COLORS[len(prefix_colors) % len(REF_COLORS)]
        )
        fig.add_trace(
            go.Scatter(
                x=x_data,
                y=y_data * 1000.0,
                mode="markers+lines",
                name=name,
                line=dict(dash="dot", shape="spline", smoothing=1.0),
                marker=dict(
                    color=color,
                    symbol="circle"
                    if "_t_v" in name
                    else "square"
                    if "_t_tr" in name
                    else "circle",
                    size=9,
                ),
            )
        )
    fig.update_layout(
        template="simple_white",
        title=title,
        xaxis_title="Time (s)",
        yaxis_title="Temperature (K)",
    )
    fig.update_xaxes(
        type="log", exponentformat="power", showexponent="all", showgrid=True
    )
    fig.update_yaxes(showgrid=True)
    if yaxis_range is not None:
        fig.update_yaxes(range=yaxis_range)
    if xaxis_range is not None:
        fig.update_xaxes(range=xaxis_range)
    fig.show()


def make_reacting_density_comparison_plot(
    *,
    results: dict[str, dict],
    reference_csv: str | Path,
    title: str,
    yaxis_range: list[float] | None = None,
    xaxis_range: list[float] | None = None,
    subsample_start_index: int = 10,
    subsample_factor: int = 10,
):
    fig = go.Figure()
    for chemistry_model, result in results.items():
        t_plot = jnp.concatenate(
            [
                result["t"][:subsample_start_index],
                result["t"][subsample_start_index::subsample_factor],
            ],
            axis=0,
        )
        n_plot = jnp.concatenate(
            [
                result["n_species"][:subsample_start_index],
                result["n_species"][subsample_start_index::subsample_factor],
            ],
            axis=0,
        )
        species = result["species"]
        for s, name in enumerate(species.names):
            fig.add_trace(
                go.Scatter(
                    x=t_plot,
                    y=n_plot[:, s],
                    mode="lines",
                    name=f"{result['display_name']} {name}",
                    line=dict(color=result["color"], dash="solid" if s == 0 else "dot"),
                )
            )
    prefix_colors = {}
    for name, x_data, y_data in iter_reference_traces(reference_csv):
        prefix = dataset_prefix(name)
        color = prefix_colors.setdefault(
            prefix, REF_COLORS[len(prefix_colors) % len(REF_COLORS)]
        )
        fig.add_trace(
            go.Scatter(
                x=x_data,
                y=y_data,
                mode="markers+lines",
                name=name,
                line=dict(dash="dot", shape="spline", smoothing=1.0),
                marker=dict(color=color, size=9),
            )
        )
    fig.update_layout(
        template="simple_white",
        title=title,
        xaxis_title="Time (s)",
        yaxis_title="Number density (1/m^3)",
    )
    fig.update_xaxes(
        type="log", exponentformat="power", showexponent="all", showgrid=True
    )
    fig.update_yaxes(
        type="log", exponentformat="power", showexponent="all", showgrid=True
    )
    if yaxis_range is not None:
        fig.update_yaxes(range=yaxis_range)
    if xaxis_range is not None:
        fig.update_xaxes(range=xaxis_range)
    fig.show()

## Heating (T > T_V)

In [4]:
print("=" * 80)
print("Casseau Heatbath — Heating")
print("=" * 80)

# --- user input start ---
T_tr_init = 10000.0
T_V_init = 1000.0
p_init_atm = 1.0

species_names = ("N2",)
energy_model = "bird"
include_electronic = True

integrator_scheme = "forward-euler"

dx = 1e-4
save_interval = 1

time_mode = "fixed"
dt = 1e-9
t_final = 1e-6
# --- user input end ---

heating_result = run_single_heatbath_case(
    species_names=species_names,
    energy_model=energy_model,
    include_electronic=include_electronic,
    T_tr_init=T_tr_init,
    T_V_init=T_V_init,
    p_init_atm=p_init_atm,
    mass_fraction_composition={"N2": 1.0},
    dx=dx,
    save_interval=save_interval,
    time_mode=time_mode,
    dt=dt,
    t_final=t_final,
    integrator_scheme=integrator_scheme,
)

Casseau Heatbath — Heating


E0412 15:11:26.756184  276707 cuda_executor.cc:1743] Could not get kernel mode driver version: ( INVALID_ARGUMENT: Version does not match the format X.Y.Z )
E0412 15:11:26.778448  275504 cuda_executor.cc:1743] Could not get kernel mode driver version: ( INVALID_ARGUMENT: Version does not match the format X.Y.Z )
E0412 15:11:30.130726  276728 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


In [5]:
make_temperature_reference_plot(
    result=heating_result,
    reference_csv=RESULTS_DIR / "casseau_figure_3_1.csv",
    title="Casseau heating comparison",
    yaxis_range=[0, 10500],
    T_eq=7623.3,
)

## Cooling (T < T_V)

In [8]:
print("=" * 80)
print("Casseau Heatbath — Cooling")
print("=" * 80)

# --- user input start ---
T_tr_init = 3000.0
T_V_init = 10000.0
p_init_atm = 1.0

species_names = ("N2",)
energy_model = "bird"
include_electronic = True

integrator_scheme = "forward-euler"

dx = 1e-4
save_interval = 1

time_mode = "fixed"
dt = 1e-8
t_final = 1e-4
# --- user input end ---

cooling_result = run_single_heatbath_case(
    species_names=species_names,
    energy_model=energy_model,
    include_electronic=include_electronic,
    T_tr_init=T_tr_init,
    T_V_init=T_V_init,
    p_init_atm=p_init_atm,
    mass_fraction_composition={"N2": 1.0},
    dx=dx,
    save_interval=save_interval,
    time_mode=time_mode,
    dt=dt,
    t_final=t_final,
    integrator_scheme=integrator_scheme,
)

Casseau Heatbath — Cooling


In [9]:
make_temperature_reference_plot(
    result=cooling_result,
    reference_csv=RESULTS_DIR / "casseau_figure_3_1_cooling.csv",
    title="Casseau cooling comparison",
    yaxis_range=[0, 10500],
    T_eq=7623.3,
)

## Case with Excitation of Electronic Energy Mode

In [3]:
print("=" * 80)
print("Casseau Heatbath — Electronic Excitation")
print("=" * 80)

# --- user input start ---
T_tr_init = 30000.0
T_V_init = 1000.0
p_init_atm = 1.0

species_names = ("N2",)
energy_model = "bird"
include_electronic = True

integrator_scheme = "rk2"

dx = 1e-4
save_interval = 1

time_mode = "fixed"
dt = 1e-9
t_final = 1e-6
# --- user input end ---

electronic_result = run_single_heatbath_case(
    species_names=species_names,
    energy_model=energy_model,
    include_electronic=include_electronic,
    T_tr_init=T_tr_init,
    T_V_init=T_V_init,
    p_init_atm=p_init_atm,
    mass_fraction_composition={"N2": 1.0},
    dx=dx,
    save_interval=save_interval,
    time_mode=time_mode,
    dt=dt,
    t_final=t_final,
    integrator_scheme=integrator_scheme,
)

Casseau Heatbath — Electronic Excitation


E0412 16:31:26.149874  300098 cuda_executor.cc:1743] Could not get kernel mode driver version: ( INVALID_ARGUMENT: Version does not match the format X.Y.Z )
E0412 16:31:26.177191  299951 cuda_executor.cc:1743] Could not get kernel mode driver version: ( INVALID_ARGUMENT: Version does not match the format X.Y.Z )


In [4]:
make_temperature_reference_plot(
    result=electronic_result,
    reference_csv=RESULTS_DIR / "casseau_figure_3_2.csv",
    title="Casseau electronic excitation comparison",
    yaxis_range=[0, 30500],
    T_eq=7623.3,
)

## VT Relaxation of Non-Reacting Multi-Species Gas (T_tr > T_V)

In [16]:
print("=" * 80)
print("Casseau Heatbath — Non-Reacting N2/N")
print("=" * 80)

# --- user input start ---
T_tr_init = 30000.0
T_V_init = 1000.0

species_names = ("N2", "N")
number_densities = {"N2": 5.0e22, "N": 5.0e22}
# Legacy benchmark setup used a fixed 1 atm initial pressure.
p_init_atm = 1.0
energy_model = "bird"
include_electronic = False

integrator_scheme = "forward-euler"

dx = 1e-4
save_interval = 1

time_mode = "fixed"
dt = 1e-9
t_final = 1e-6
# --- user input end ---

multispecies_heating_result = run_single_heatbath_case(
    species_names=species_names,
    energy_model=energy_model,
    include_electronic=include_electronic,
    T_tr_init=T_tr_init,
    T_V_init=T_V_init,
    number_densities=number_densities,
    p_init_atm=p_init_atm,
    dx=dx,
    save_interval=save_interval,
    time_mode=time_mode,
    dt=dt,
    t_final=t_final,
    integrator_scheme=integrator_scheme,
)

Casseau Heatbath — Non-Reacting N2/N


In [17]:
make_temperature_reference_plot(
    result=multispecies_heating_result,
    reference_csv=RESULTS_DIR / "casseau_figure_3_3.csv",
    title="Casseau non-reacting N2/N comparison",
    yaxis_range=[0, 30500],
    T_eq=7623.3,
)

## VT Relaxation of Non-Reacting Multispecies Gas (N2, O2) (T_tr < T_V)

Casseau correlates both VT and VV relaxation here. VV relaxation is still not implemented in this model, so the comparison remains limited to the VT part.

In [18]:
print("=" * 80)
print("Casseau Heatbath — Non-Reacting N2/O2 Cooling")
print("=" * 80)

# --- user input start ---
T_tr_init = 5000.0
T_V_init = 30000.0

species_names = ("N2", "O2")
number_densities = {"N2": 5.0e22, "O2": 5.0e22}
# Legacy benchmark setup used a fixed 1 atm initial pressure.
p_init_atm = 1.0
energy_model = "bird"
include_electronic = False

integrator_scheme = "rk2"

dx = 1e-4
save_interval = 1

time_mode = "fixed"
dt = 1e-9
t_final = 1e-6
# --- user input end ---

multispecies_cooling_result = run_single_heatbath_case(
    species_names=species_names,
    energy_model=energy_model,
    include_electronic=include_electronic,
    T_tr_init=T_tr_init,
    T_V_init=T_V_init,
    number_densities=number_densities,
    p_init_atm=p_init_atm,
    dx=dx,
    save_interval=save_interval,
    time_mode=time_mode,
    dt=dt,
    t_final=t_final,
    integrator_scheme=integrator_scheme,
)

Casseau Heatbath — Non-Reacting N2/O2 Cooling


In [19]:
make_temperature_reference_plot(
    result=multispecies_cooling_result,
    reference_csv=RESULTS_DIR / "casseau_figure_3_4.csv",
    title="Casseau non-reacting N2/O2 cooling comparison",
    yaxis_range=[0, 30500],
    T_eq=7623.3,
)

## Relaxation of a Chemically Reacting Mixture

In [14]:
print("=" * 80)
print("Casseau Heatbath — Reacting Mixture, Thermal Non-Equilibrium")
print("=" * 80)

# --- user input start ---
T_tr_init = 30000.0
T_V_init = 1000.0

species_names = ("N2", "N")
number_densities = {"N2": 5.0e22, "N": 5.0e22}
energy_model = "bird"
include_electronic = False
comparison_models = [
    {"chemistry_model": "park_pref", "reaction_set": "n2_park"},
    {"chemistry_model": "park_nonpref", "reaction_set": "n2_park"},
    {"chemistry_model": "cvdv_qp", "reaction_set": "n2_qk"},
]

dx = 1e-4
save_interval = 1

time_mode = "two_phase"
dt_fine = 1e-9
dt_coarse = 1e-7
t_threshold = 1e-6
t_final = 1e-3
# --- user input end ---

reacting_noneq_results = run_reacting_comparison(
    species_names=species_names,
    number_densities=number_densities,
    T_tr_init=T_tr_init,
    T_V_init=T_V_init,
    energy_model=energy_model,
    include_electronic=include_electronic,
    comparison_models=comparison_models,
    dx=dx,
    save_interval=save_interval,
    time_mode=time_mode,
    t_final=t_final,
    dt_fine=dt_fine,
    dt_coarse=dt_coarse,
    t_threshold=t_threshold,
)
print_comparison_summary(reacting_noneq_results)

Casseau Heatbath — Reacting Mixture, Thermal Non-Equilibrium
Completed park_pref: reaction_set=n2_park, energy_model=bird
Completed park_nonpref: reaction_set=n2_park, energy_model=bird
Completed cvdv_qp: reaction_set=n2_qk, energy_model=bird

=== Park preferential (park_pref) ===
Included reactions:
  N2 + N2 -> 2N + N2
Excluded reactions:

=== Park nonpreferential (park_nonpref) ===
Included reactions:
  N2 + N2 -> 2N + N2
Excluded reactions:

=== CVDV-QP (cvdv_qp) ===
Included reactions:
  N2 + N2 -> 2N + N2
Excluded reactions:


In [15]:
make_reacting_temperature_comparison_plot(
    results=reacting_noneq_results,
    reference_csv=RESULTS_DIR / "casseau_figure_3_5_temperature.csv",
    title="Casseau reacting mixture temperature comparison",
    yaxis_range=[3.0e3, 3.2e4],
    xaxis_range=[-9, -2],
)

make_reacting_density_comparison_plot(
    results=reacting_noneq_results,
    reference_csv=RESULTS_DIR / "casseau_figure_3_5_density.csv",
    title="Casseau reacting mixture density comparison",
    xaxis_range=[-9, -2],
)

## Relaxation of Chemically Reacting Mixture at Thermal Equilibrium

In [16]:
print("=" * 80)
print("Casseau Heatbath — Reacting Mixture at Thermal Equilibrium")
print("=" * 80)

# --- user input start ---
T_tr_init = 30000.0
T_V_init = 30000.0

species_names = ("N2", "N")
number_densities = {"N2": 5.0e22, "N": 5.0e22}
energy_model = "bird"
include_electronic = False
comparison_models = [
    {"chemistry_model": "park_pref", "reaction_set": "n2_park"},
    {"chemistry_model": "park_nonpref", "reaction_set": "n2_park"},
    {"chemistry_model": "cvdv_qp", "reaction_set": "n2_qk"},
]

dx = 1e-4
save_interval = 1

time_mode = "two_phase"
dt_fine = 1e-9
dt_coarse = 1e-7
t_threshold = 1e-6
t_final = 1e-4
# --- user input end ---

reacting_eq_results = run_reacting_comparison(
    species_names=species_names,
    number_densities=number_densities,
    T_tr_init=T_tr_init,
    T_V_init=T_V_init,
    energy_model=energy_model,
    include_electronic=include_electronic,
    comparison_models=comparison_models,
    dx=dx,
    save_interval=save_interval,
    time_mode=time_mode,
    t_final=t_final,
    dt_fine=dt_fine,
    dt_coarse=dt_coarse,
    t_threshold=t_threshold,
)
print_comparison_summary(reacting_eq_results)

Casseau Heatbath — Reacting Mixture at Thermal Equilibrium
Completed park_pref: reaction_set=n2_park, energy_model=bird


E0412 13:13:32.816668  240495 pjrt_stream_executor_client.cc:2091] Execution of replica 0 failed: INTERNAL: CpuCallback error calling callback: Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/hhoechter/tum/jaxfluids_internship/.venv/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
  File "/home/hhoechter/tum/jaxfluids_internship/.venv/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
  File "/home/hhoechter/tum/jaxfluids_internship/.venv/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 758, in start
  File "/home/hhoechter/tum/jaxfluids_internship/.venv/lib/python3.11/site-packages/tornado/platform/asyncio.py", line 211, in start
  File "/usr/lib/python3.11/asyncio/base_events.py", line 604, in run_forever
  File "/usr/lib/python3.11/asyncio/base_events.py", line 1909, in _run_once
  File "/usr/lib/python

JaxRuntimeError: INTERNAL: CpuCallback error calling callback: Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/hhoechter/tum/jaxfluids_internship/.venv/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
  File "/home/hhoechter/tum/jaxfluids_internship/.venv/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
  File "/home/hhoechter/tum/jaxfluids_internship/.venv/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 758, in start
  File "/home/hhoechter/tum/jaxfluids_internship/.venv/lib/python3.11/site-packages/tornado/platform/asyncio.py", line 211, in start
  File "/usr/lib/python3.11/asyncio/base_events.py", line 604, in run_forever
  File "/usr/lib/python3.11/asyncio/base_events.py", line 1909, in _run_once
  File "/usr/lib/python3.11/asyncio/events.py", line 80, in _run
  File "/home/hhoechter/tum/jaxfluids_i

In [ ]:
make_reacting_temperature_comparison_plot(
    results=reacting_eq_results,
    reference_csv=RESULTS_DIR / "casseau_figure_3_6_temperature.csv",
    title="Casseau reacting equilibrium temperature comparison",
    yaxis_range=[5.0e3, 3.2e4],
    xaxis_range=[-9, -3],
)

make_reacting_density_comparison_plot(
    results=reacting_eq_results,
    reference_csv=RESULTS_DIR / "casseau_figure_3_6_density.csv",
    title="Casseau reacting equilibrium density comparison",
    xaxis_range=[-9, -3],
)

## Chemically Reacting Air

In [ ]:
print("=" * 80)
print("Casseau Heatbath — Chemically Reacting Air")
print("=" * 80)

# --- user input start ---
T_tr_init = 10000.0
T_V_init = 10000.0

species_names = ("N2", "N", "O2", "O", "NO")
n_tot = 4.625e22
number_densities = {
    "N2": 0.78 * n_tot,
    "N": 0.0,
    "O2": 0.21 * n_tot,
    "O": 0.0,
    "NO": 0.0,
}
energy_model = "bird"
include_electronic = False
comparison_models = [
    {"chemistry_model": "park_pref", "reaction_set": "park"},
    {"chemistry_model": "park_nonpref", "reaction_set": "park"},
    {"chemistry_model": "cvdv_qp", "reaction_set": "qk"},
]

dx = 1e-4
save_interval = 1

time_mode = "two_phase"
dt_fine = 1e-9
dt_coarse = 1e-9
t_threshold = 1e-6
t_final = 1e-4
# --- user input end ---

reacting_air_results = run_reacting_comparison(
    species_names=species_names,
    number_densities=number_densities,
    T_tr_init=T_tr_init,
    T_V_init=T_V_init,
    energy_model=energy_model,
    include_electronic=include_electronic,
    comparison_models=comparison_models,
    dx=dx,
    save_interval=save_interval,
    time_mode=time_mode,
    t_final=t_final,
    dt_fine=dt_fine,
    dt_coarse=dt_coarse,
    t_threshold=t_threshold,
)
print_comparison_summary(reacting_air_results)

In [ ]:
make_reacting_temperature_comparison_plot(
    results=reacting_air_results,
    reference_csv=RESULTS_DIR / "casseau_figure_3_7_temperature.csv",
    title="Casseau reacting air temperature comparison",
    yaxis_range=[5.0e3, 1.1e4],
    xaxis_range=[-9, -3],
)

make_reacting_density_comparison_plot(
    results=reacting_air_results,
    reference_csv=RESULTS_DIR / "casseau_figure_3_7_density.csv",
    title="Casseau reacting air density comparison",
    xaxis_range=[-9, -3],
)